In [ ]:
from utils import families, make_instance, grade, evaluate, paper_id, load_instances
from datasets import load_dataset

DATASET = "amphora/math-intuition-20260905-408-easy-30"
ds = load_dataset(DATASET)['train']

# instances generated once by vllm_gen_eval.ipynb and pickled with utils.save_instances();
# load_instances() refuses a cache whose rows or generators no longer match.
instances = load_instances(ds, DATASET)
print(f"{len(ds)} rows, {len(instances)} cached instances")

In [ ]:
import os
# this is not necessary for most cases.
os.environ["HF_HOME"] = "/home/work/onelineai/hf_cache"
os.environ["HF_HUB_CACHE"] = "/home/work/onelineai/hf_cache/hub"

from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"
llm = LLM(model=MODEL, tensor_parallel_size=8)

In [ ]:
from tqdm import tqdm
tokenizer = llm.get_tokenizer()
qrys = []

prompt = "Solve the provided question. Do not use code execution to solve."

for row in tqdm(ds):
    question = row['question']

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    qrys.append(text)

assert len(qrys) == len(instances)

In [ ]:
sampling_params = SamplingParams(temperature=1.0)
output = llm.generate(qrys, sampling_params)

In [ ]:
# Grade every reply with its family's own verifier, then dump everything to jsonl so the run survives the kernel.
import json, os
from utils import CACHE_DIR

results = []
for row, (fam, inst), out in zip(ds, instances, output):
    reply = out.outputs[0].text
    try:
        correct, reason, answer = grade(fam, inst, reply)
    except Exception as e:
        correct, reason, answer = False, f"grader raised {type(e).__name__}: {e}", None
    results.append({"id": row["id"], "paper": paper_id(row["paper"]), "preset": row["preset"], "seed": row["seed"],
                    "model": MODEL, "correct": correct, "reason": reason, "answer": answer, "reply": reply})

n_ok = sum(r["correct"] for r in results)
print(f"{MODEL}: {n_ok}/{len(results)} correct, {n_ok/len(results):.2%}")

OUT_PATH = os.path.join(CACHE_DIR, f"results_{DATASET.split('/')[-1]}_{MODEL.split('/')[-1]}.jsonl")
with open(OUT_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r, default=str) + "\n")
print("wrote", OUT_PATH)